# External validation: do DEME's dimensions track *human* moral labels?

The paper argues moral evaluation is genuinely multi-dimensional. The companion AITA notebook shows DEME's own vectors are not scalar-reducible, but that is self-referential. This notebook is the stronger test: do DEME's dimensions recover **independent, human-annotated** moral structure?

We use the **Moral Foundations Reddit Corpus** (MFRC; Trager et al.) — Reddit comments hand-labeled by trained annotators for moral foundations (Care, Equality, Proportionality, Authority, Loyalty, Purity). For a stratified sample of 280 comments we scored each on DEME's ten dimensions via the NRP managed-LLM panel (`score_mfrc_nrp.py`), then correlate the DEME scores against the human foundation labels. The committed `deme_mfrc_scores.jsonl` holds the derived scores + human label fractions (no raw comment text; source: HF `USC-MOLA-Lab/MFRC`).

**Pre-registered alignments** (named before looking): `care_protection`↔Care, `fairness_equity`↔Equality/Proportionality, `legitimacy_trust`↔Authority, `vow_fidelity`↔Loyalty. Purity has no DEME analog (a negative control).

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

D = [json.loads(l) for l in open('deme_mfrc_scores.jsonl', encoding='utf-8')]
DIMS = ['physical_harm', 'rights_respect', 'fairness_equity', 'autonomy_consent',
        'legitimacy_trust', 'epistemic_quality', 'care_protection', 'vow_fidelity',
        'third_party_externality', 'repair_residue']
FOUND = ['Care', 'Equality', 'Proportionality', 'Authority', 'Loyalty', 'Purity']
V = np.array([[r['vec'][k] for k in DIMS] for r in D])      # DEME scores (LLM panel)
H = np.array([[r['frac'][f] for f in FOUND] for r in D])    # human foundation fractions
print(f'{len(D)} MFRC comments | DEME vectors {V.shape} | human foundations {H.shape}')

## Pre-registered alignments hold

In [ ]:
pairs = [('care_protection', 'Care'), ('fairness_equity', 'Equality'),
         ('fairness_equity', 'Proportionality'), ('legitimacy_trust', 'Authority'),
         ('vow_fidelity', 'Loyalty')]
rhos, ps = [], []
for dim, f in pairs:
    rho, p = spearmanr(V[:, DIMS.index(dim)], H[:, FOUND.index(f)])
    rhos.append(rho); ps.append(p)
    print(f'  {dim:16s} ~ {f:14s} rho={rho:+.3f}  p={p:.1e}')
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(range(len(pairs)), rhos, color='tab:green')
ax.set_xticks(range(len(pairs)))
ax.set_xticklabels([f'{d}\n~ {f}' for d, f in pairs], fontsize=8)
ax.set_ylabel('Spearman ρ  (DEME dim vs human label)')
ax.set_title('DEME dimensions recover independent human moral foundations')
for i, r in enumerate(rhos):
    ax.text(i, r + 0.01, f'{r:.2f}', ha='center', fontsize=9)
plt.tight_layout(); plt.show()

## The full picture: a diagonal structure

Correlating every DEME dimension against every human foundation, the pre-registered pairs (boxed) light up — the representation aligns where it should and stays quiet where it shouldn't (Purity, with no DEME analog, correlates weakly with everything).

In [ ]:
M = np.array([[spearmanr(V[:, i], H[:, j])[0] for j in range(len(FOUND))]
              for i in range(len(DIMS))])
fig, ax = plt.subplots(figsize=(7, 7.5))
im = ax.imshow(M, cmap='RdBu_r', vmin=-0.5, vmax=0.5, aspect='auto')
ax.set_xticks(range(len(FOUND))); ax.set_xticklabels(FOUND, rotation=45, ha='right')
ax.set_yticks(range(len(DIMS))); ax.set_yticklabels(DIMS)
for i in range(len(DIMS)):
    for j in range(len(FOUND)):
        ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=7,
                color='white' if abs(M[i, j]) > 0.3 else 'black')
for dim, f in pairs:
    ax.add_patch(plt.Rectangle((FOUND.index(f) - 0.5, DIMS.index(dim) - 0.5), 1, 1,
                               fill=False, edgecolor='lime', lw=2.5))
fig.colorbar(im, ax=ax, shrink=0.6, label='Spearman ρ')
ax.set_title('DEME dimensions × human moral foundations\n(green = pre-registered alignment)')
plt.tight_layout(); plt.show()

## The diagonal is the argmax

The strongest test: for each aligned dimension, is its intended human foundation the one it correlates with *most* — not just significantly?

In [ ]:
for dim, f in [('care_protection', 'Care'), ('legitimacy_trust', 'Authority'),
               ('vow_fidelity', 'Loyalty'), ('fairness_equity', 'Equality')]:
    rr = sorted(((spearmanr(V[:, DIMS.index(dim)], H[:, j])[0], FOUND[j])
                 for j in range(len(FOUND))), reverse=True)
    hit = 'YES' if rr[0][1] == f else 'no'
    top3 = ', '.join(f'{name} {r:.2f}' for r, name in rr[:3])
    print(f'  {dim:16s}: argmax→ {top3}   [intended {f}: {hit}]')

## Conclusion

DEME's dimensions are not arbitrary coordinates: on 280 independently human-annotated MFRC comments, each pre-registered dimension correlates with its intended moral foundation at ρ≈0.47–0.49 (all p<1e-16), and the intended foundation is the *argmax* in every case. This is an **external, human-grounded** validation of the representation — exactly the test a reviewer asks for when the worry is that the multi-dimensional structure is self-referential. It complements (does not replace) the in-corpus probe in the AITA notebook, and it is reproducible from `deme_mfrc_scores.jsonl` (analysis) plus `score_mfrc_nrp.py` + MFRC (re-deriving the scores).